# The notebook bridge — canvas as DUT, notebook as bench

The browser is where you *shape* a circuit; this notebook is where you *interrogate* it.
`photonflux.nb` connects to the web UI's live schematic mirror, so both sides always see
the same document: edit on the canvas and the notebook sees it instantly, push from here
and the canvas updates (undoably — `Cmd+Z` reverts a notebook edit like any other).

Setup: `python webapp/server.py`, open http://127.0.0.1:8642 in a browser, run cells.

In [ ]:
from photonflux.nb import Session, Builder

s = Session()          # http://127.0.0.1:8642 (or Session(url=...))
s

In [ ]:
sch = s.pull()         # the live schematic from the browser's active tab
sch

## Run the canvas

`s.run()` with no arguments is the Run button's twin: the live schematic with the
analysis configured in its toolbar. Or pass an explicit analysis — every toolbar
mode has a helper (`transient`, `dc`, `dcsweep`, `ac`, `noise`, `pulse`), all
accepting SPICE-style SI suffixes (`"20n"`, `"50G"`). Progress streams from the
server; Ctrl-C (interrupt kernel) cancels the run server-side.

In [ ]:
res = s.run()
res.plot()
res

## Edit the canvas from here

Parameter writes are read-modify-write against the mirror revision, so they never
clobber a canvas edit that races them — and they land in the browser's undo stack.

In [ ]:
ref, inst = s.selected() or (None, None)   # whatever you clicked in the browser
print(ref, "->", inst["type"] if inst else None)

# s["LAS1.power"] = "2m"                   # watch the canvas headline update

## The reactive bench

`s.watch()` yields every time the canvas changes. Drag a coupling, nudge a length —
the loop below re-sweeps the spectrum and redraws in place, like a spectrum analyzer
bolted to the schematic. Interrupt the kernel to stop.

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import clear_output

# for sch in s.watch(initial=True, debounce=0.3):
#     res = s.dcsweep("*", "wavelength_nm", 1307.5, 1312.5, points=1001)
#     clear_output(wait=True)
#     res.plot()
#     plt.show()

## Build a schematic programmatically

An all-pass microring from parts, pushed onto the canvas for hand-refinement —
parametric generation is a loop here, and layout aesthetics stay a human job there.
At critical coupling ($\kappa^2 = 1 - a^2$, here 0.088 against the 20 dB/cm loop
loss) the through-port notch bottoms out.

In [ ]:
b = Builder()
las = b.add("cw_laser", power="1m", wavelength_nm=1310)
gnd = b.add("ground")
cpl = b.add("dir_coupler", coupling=0.088)
wg  = b.add("waveguide", length_m="199.775u", loss_dB_cm=20.0,
            neff=2.4, n_group=4.0, center_wavelength_nm=1310.0)
tt  = b.add("opt_term")
b.wire(las.p1, cpl.p1); b.wire(las.p2, gnd.p1)
b.wire(cpl.p4, wg.p2);  b.wire(wg.p1, cpl.p2)
b.wire(cpl.p3, tt.p1)
b.probe(cpl.p3, name="thru")

s.push(b, title="ring bench (from notebook)")
res = s.dcsweep("*", "wavelength_nm", 1307.5, 1312.5, points=1001)
ax = res.plot()
ax.set_yscale("log")

## Sweep beyond the toolbar

Anything the toolbar can't express is a Python loop over the same machinery —
here the notch depth vs. coupling, i.e. *finding* critical coupling numerically.

In [ ]:
import numpy as np

depths = []
couplings = np.linspace(0.02, 0.30, 8)
for k2 in couplings:
    s["DIR1.coupling"] = float(k2)
    r = s.dcsweep("*", "wavelength_nm", 1309.5, 1310.5, points=501)
    depths.append(10 * np.log10(r["thru"].min() / r["thru"].max()))

plt.plot(couplings, depths, "o-")
plt.xlabel("coupler $\\kappa^2$"); plt.ylabel("notch depth [dB]")
plt.title("critical coupling, found empirically"); plt.grid(alpha=0.3)

## Provenance

Freeze the exact document an analysis ran on, so the result is reproducible after
the canvas moves on: `s.snapshot("before_tuning")` writes it next to the notebook;
`Schematic.load(...)` + `s.run(schematic=...)` replays it.